## 02 — Bulk fixture ingest from Sportmonks API (batched)

Production notebook. Fetches detailed data for **all** season fixtures using the Sportmonks multi-fixture endpoint (up to 50 IDs per request). Includes retry logic and a detailed ingestion log. The output is a directory of JSONs in the Landing Zone ready for notebook 03.


### 1. Imports

In addition to standard libraries, imports `HTTPAdapter` and `Retry` from `urllib3` — enables automatic request retries on network errors and rate-limiting responses.


In [ ]:
import json
import os
from datetime import datetime, timezone
import pyspark

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry 

### 2. Configuration 
 
`TOKEN` is retrieved from Databricks Secret Scope. `BASE_PATH` is the root of the Unity Catalog Volume for this data layer.


In [ ]:
LEAGUE_ID = 45
SEASON_ID = 26097

TOKEN = dbutils.secrets.get(
    scope = "sportmonks",
    key = "api-token")

BASE_PATH = (
    "/Volumes/wsl_analytics/landing/sportmonks_raw" # change if needed
)

### 3. Load fixture list 

Reads the fixture list file produced in the previous ingestion step (notebook 01 or equivalent). The `*/` wildcard in the path handles the `ingestion_date=…` partition without requiring a hardcoded date.


In [ ]:
fixture_list_path = (
    f"{BASE_PATH}/fixture_lists/"
    f"league_id={LEAGUE_ID}/"
    f"season_id={SEASON_ID}/*/fixtures.json"
)

fixture_list_df = (
    spark.read
    .option("multiLine", "true")
    .json(fixture_list_path)
)

display(fixture_list_df)

data ingestion_timestamp league_id season_id List(List(null, null, null, true, true, 19499028, 45, 1/1, 90, Chelsea W vs Manchester City W, List(List(462, 0, female, 59583, https://cdn.sportmonks.com/images/soccer/teams/31/59583.png, 2026-05-16 12:00:00, List(home, 1, true), Chelsea W, false, null, 1, domestic, 666), List(462, null, female, 59584, https://cdn.sportmonks.com/images/soccer/teams/0/59584.png, 2026-05-31 14:00:00, List(away, 2, false), Manchester City W, false, null, 1, domestic, 338908)), false, Chelsea W won after full-time., 380511, List(List(1ST_HALF, 19499028, 16899123, 59583, List(1, home), 1), List(2ND_HALF_ONLY, 19499028, 16899916, 59584, List(1, away), 48996), List(CURRENT, 19499028, 16899121, 59583, List(2, home), 1525), List(CURRENT, 19499028, 16899122, 59584, List(1, away), 1525), List(2ND_HALF_ONLY, 19499028, 16899915, 59583, List(1, home), 48996), List(1ST_HALF, 19499028, 16899124, 59584, List(0, away), 1), List(2ND_HALF, 19499028, 16899125, 59583, List(2, home), 2), List(2ND_HALF, 19499028, 16899126, 59584, List(1, away), 2)), 26097, 1, 77477422, 2025-09-05 18:30:00, 1757097000, List(FT, 5, Full Time, FT, FT), 5, 321614), List(null, null, null, true, true, 19499029, 45, 1/1, 90, Arsenal W vs London City Lionesses W, List(List(462, null, female, 132782, https://cdn.sportmonks.com/images/soccer/teams/14/132782.png, 2026-05-16 12:00:00, List(away, 6, false), London City Lionesses W, false, null, 1, domestic, 343366), List(462, null, female, 62847, https://cdn.sportmonks.com/images/soccer/teams/31/62847.png, 2026-05-16 12:00:00, List(home, 2, true), Arsenal W, false, null, 1, domestic, 204)), false, Arsenal W won after full-time., 380511, List(List(1ST_HALF, 19499029, 16902752, 62847, List(2, home), 1), List(2ND_HALF, 19499029, 16902753, 132782, List(1, away), 2), List(1ST_HALF, 19499029, 16902751, 132782, List(1, away), 1), List(CURRENT, 19499029, 16902749, 132782, List(1, away), 1525), List(2ND_HALF, 19499029, 16902754, 62847, List(4, home), 2), List(CURRENT, 19499029, 16902750, 62847, List(4, home), 1525), List(2ND_HALF_ONLY, 19499029, 16905977, 62847, List(2, home), 48996), List(2ND_HALF_ONLY, 19499029, 16905978, 132782, List(0, away), 48996)), 26097, 1, 77477422, 2025-09-06 12:30:00, 1757161800, List(FT, 5, Full Time, FT, FT), 5, 204), List(null, null, null, true, true, 19499030, 45, 1/1, 90, Manchester United W vs Leicester W, List(List(462, null, female, 228655, https://cdn.sportmonks.com/images/soccer/teams/15/228655.png, 2026-05-16 12:00:00, List(home, 5, true), Manchester United W, false, null, 1, domestic, 1208), List(462, null, female, 228634, https://cdn.sportmonks.com/images/soccer/teams/26/228634.png, 2026-05-16 12:00:00, List(away, 8, false), Leicester W, false, null, 1, domestic, 117)), false, Manchester United W won after full-time., 380511, List(List(CURRENT, 19499030, 16915391, 228634, List(0, away), 1525), List(2ND_HALF, 19499030, 16915405, 228655, List(4, home), 2), List(1ST_HALF, 19499030, 16915397, 228634, List(0, away), 1), List(CURRENT, 19499030, 16915394, 228655, List(4, home), 1525), List(1ST_HALF, 19499030, 16915401, 228655, List(2, home), 1), List(2ND_HALF, 19499030, 16915403, 228634, List(0, away), 2), List(2ND_HALF_ONLY, 19499030, 16917306, 228655, List(2, home), 48996), List(2ND_HALF_ONLY, 19499030, 16917307, 228634, List(0, away), 48996)), 26097, 1, 77477422, 2025-09-07 11:00:00, 1757242800, List(FT, 5, Full Time, FT, FT), 5, 206), List(null, null, null, true, true, 19499031, 45, 1/1, 90, Tottenham W vs West Ham W, List(List(462, null, female, 228102, https://cdn.sportmonks.com/images/soccer/teams/6/228102.png, 2026-05-16 12:00:00, List(away, 7, false), West Ham W, false, WHU, 1, domestic, 988), List(462, null, female, 132699, https://cdn.sportmonks.com/images/soccer/teams/27/132699.png, 2026-08-12 18:00:00, List(home, 6, true), Tottenham W, false, null, 1, domestic, 832)), false, Tottenham W won after full-time., 380511, List(List(2ND_HALF_ONLY, 19499031, 16917311, 228

### 4. Extract fixture IDs 
 
Explodes the `data` array (list of fixtures returned by the API), then selects only the required columns. `dropDuplicates` guards against duplicates that could arise from multiple writes of the fixture list.


In [ ]:
from pyspark.sql import functions as F

fixture_ids_df = (
    fixture_list_df
    .select(
        F.explode("data").alias("fixture")
    )
    .select(
        F.col("fixture.id").alias("fixture_id"),
        F.col("fixture.name").alias("fixture_name"),
        F.col("fixture.starting_at").alias("starting_at")
    )
    .dropDuplicates(["fixture_id"])
)

display(fixture_ids_df)

fixture_id,fixture_name,starting_at
19499036,London City Lionesses W vs Manchester United W,2025-09-14 11:00:00
19499067,Aston Villa W vs Everton W,2025-11-02 12:00:00
19499111,Manchester City W vs Chelsea W,2026-02-01 14:30:00
19499133,Manchester City W vs Tottenham W,2026-03-21 12:00:00
19499144,Tottenham W vs Manchester United W,2026-04-26 11:00:00
19499150,Aston Villa W vs West Ham W,2026-05-04 12:00:00
19499044,Brighton W vs West Ham W,2025-09-21 11:00:00
19499089,Manchester City W vs Aston Villa W,2025-12-14 11:55:00
19499101,Everton W vs Brighton W,2026-01-23 19:00:00
19499124,West Ham W vs Manchester United W,2026-03-18 19:15:00


### 5. Count fixtures
  
Verifies the list contains the expected number of fixtures in the season (132 for WSL 2025-26).


In [ ]:
fixture_ids_df.count()

132

### 6. Order by date

Displays fixtures chronologically — useful for manually verifying the list is complete and gap-free.


In [ ]:
display(fixture_ids_df.orderBy("starting_at"))

fixture_id,fixture_name,starting_at
19499028,Chelsea W vs Manchester City W,2025-09-05 18:30:00
19499029,Arsenal W vs London City Lionesses W,2025-09-06 12:30:00
19499031,Tottenham W vs West Ham W,2025-09-07 11:00:00
19499033,Liverpool W vs Everton W,2025-09-07 11:00:00
19499032,Brighton W vs Aston Villa W,2025-09-07 11:00:00
19499030,Manchester United W vs Leicester W,2025-09-07 11:00:00
19499034,Manchester City W vs Brighton W,2025-09-12 18:30:00
19499035,West Ham W vs Arsenal W,2025-09-12 18:30:00
19499036,London City Lionesses W vs Manchester United W,2025-09-14 11:00:00
19499039,Leicester W vs Liverpool W,2025-09-14 11:00:00


### 7. Collect fixture IDs 

`.collect()` pulls all IDs to the driver as a Python list. Safe because we're only collecting IDs (integers) — no large data transfer.


In [ ]:
fixture_ids = [
    row["fixture_id"]
    for row in fixture_ids_df.collect()
]

len(fixture_ids)

132

### 8. Preview 

Preview of the first 10 IDs — quick format verification.


In [ ]:
fixture_ids[:10]

[19499036,
 19499067,
 19499111,
 19499133,
 19499144,
 19499150,
 19499044,
 19499089,
 19499101,
 19499124]

### 9. Define includes 
  
List of variables to embed in each multi-fixture request. Slightly narrower than notebook 01 — omits e.g. `comments` and `tvStations` which are not needed for analysis.


In [ ]:
FULL_INCLUDES = [
    "state;"
    "participants;"
    "scores;"
    "events;"
    "timeline;"
    "statistics;"
    "lineups;"
    "periods;"
    "formations;"
    "referees;"
    "sidelined;"
    "metadata;"
    "coaches;"
    "round;"
    "venue;"
    "league;"
    "pressure;"
]

### 10. Retry strategy

Configures an HTTPAdapter with automatic retry (5 attempts) for error codes 429 (rate limit), 500, 502, 503, 504. `backoff_factor=1` gives: 1 s, 2 s, 4 s, 8 s, 16 s delays between attempts. `respect_retry_after_header=True` honours the `Retry-After` header sent by Sportmonks during rate-limiting.


In [ ]:
retry_strategy = Retry(
    total = 5,
    backoff_factor = 1,
    status_forcelist = [
        429,
        500,
        502,
        503,
        504
    ],
    allowed_methods = ["GET"],
    respect_retry_after_header = True
)

adapter = HTTPAdapter(
    max_retries = retry_strategy
)

session = requests.Session()

session.mount(
    "https://",
    adapter
)

### 11. Batch helper
 
Generator `chunks()` splits the ID list into batches of a given size. The Sportmonks multi-fixture endpoint accepts at most 50 IDs per request — hence the default `size=50`.


In [ ]:
def chunks(items, size = 50):
    for i in range(0, len(items), size):
        yield items[i:i + size]

### 12. Preview batches

Prints fixture counts per batch — verifies none exceeds the 50-ID limit.


In [ ]:
batches = list(chunks(fixture_ids, 50))
[len(batch) for batch in batches]batches = list(
    chunks(
        fixture_ids,
        size=50
    )
)

print("Liczba batchy:", len(batches))

for i, batch in enumerate(batches, start=1):
    print(
        f"Batch {i}: {len(batch)} fixtures"
    )


[50, 50, 32]

### 13. Set output directory 

Creates the output directory with Hive-style partitioning. Uses `dbutils.fs.mkdirs()` (rather than `os.makedirs`) because the path points to a Unity Catalog Volume, not the local filesystem.


In [ ]:
ingestion_timestamp = datetime.now(timezone.utc)
ingestion_data = (
    ingestion_timestamp
    .date()
    .isoformat()
)

output_dir = (
    f"{BASE_PATH}/fixture_details/"
    f"league_id={LEAGUE_ID}/"
    f"season_id={SEASON_ID}/"
    f"ingestion_date = {ingestion_data}"
)

os.makedirs(
    output_dir,
    exist_ok = True
)

print(output_dir)

/Volumes/wsl_analytics/landing/sportmonks_raw/fixture_details/league_id=45/season_id=26097/ingestion_date = 2026-08-13


### 14. Fetch function 
 
`get_fixture_batch()` builds the multi-fixture URL from comma-joined IDs, sends a GET request via the retry-enabled session, and returns the parsed JSON. On HTTP error it prints the status and response fragment before raising.


In [ ]:
def get_fixture_batch(fixture_ids):

    ids_string = ",".join(
        str(fixture_id)
        for fixture_id in fixture_ids
    )

    url = (
        "https://api.sportmonks.com/"
        f"v3/football/fixtures/multi/{ids_string}"
    )

    params = {
        "api_token": TOKEN,
        "include": FULL_INCLUDES
    }

    response = session.get(
        url,
        params=params,
        timeout=(10, 60)
    )

    if not response.ok:
        print(
            "HTTP status:",
            response.status_code
        )
        print(
            "SportMonks response:",
            response.text[:1000]
        )

    response.raise_for_status()

    return response.json()

### 15. Test fetch (2 fixtures) 

Before the main loop, tests the endpoint on a small sample (2 fixtures) to confirm authentication and includes work correctly.


In [ ]:
test_batch = fixture_ids[:2]
test_response = get_fixture_batch(
    test_batch
)

dict_keys(['data', 'subscription', 'rate_limit', 'timezone'])
Liczba fixtures: 2


### 16. Inspect response keys 

Prints top-level API response keys.


In [ ]:
test_response.keys()

dict_keys(['data', 'subscription', 'rate_limit', 'timezone'])

### 17. Count returned fixtures 

Checks how many fixtures were returned in the test — should be 2.


In [ ]:
len(test_response["data"])

2

### 18. Inspect first fixture 

Prints key fields from the first test fixture: ID, name, and array sizes for events, statistics, lineups, and pressure records. `pressure` is of particular interest — an empty array means this fixture will have no pressure data in the silver layer.


In [ ]:
fixture = test_response["data"][0]

print("fixture_id:", fixture["id"])
print("name:", fixture["name"])

print(
    "events:",
    len(fixture.get("events", []))
)

print(
    "statistics:",
    len(fixture.get("statistics", []))
)

print(
    "lineups:",
    len(fixture.get("lineups", []))
)

print(
    "pressure:",
    len(fixture.get("pressure", []))
)

fixture_id: 19499036
name: London City Lionesses W vs Manchester United W
events: 17
statistics: 78
lineups: 40
pressure: 194


### 19. Preview pressure records 

Displays the first 3 pressure records — verifies the structure (`id`, `fixture_id`, `participant_id`, `minute`, `pressure`).


In [ ]:
fixture.get("pressure", [])[:3]

[{'id': 6662762376,
  'fixture_id': 19499036,
  'participant_id': 228655,
  'minute': 1,
  'pressure': 0},
 {'id': 6663273998,
  'fixture_id': 19499036,
  'participant_id': 228655,
  'minute': 20,
  'pressure': 0.22},
 {'id': 6663273999,
  'fixture_id': 19499036,
  'participant_id': 228655,
  'minute': 21,
  'pressure': 0}]

### 20-21. Re-check events & pressure 
  
Additional printout of event and pressure record counts for the test fixture — confirms the includes worked.


In [ ]:
print(
    "events:",
    len(fixture.get("events", []))
)

events: 17


In [ ]:
print(
    "pressure:",
    len(fixture.get("pressure", []))
)

pressure: 194


### 22-23. Final batch setup 

Final version of batch and output directory setup


In [ ]:
batches = list(
    chunks(
        fixture_ids,
        size=50
    )
)

print("Number of batches:", len(batches))

for i, batch in enumerate(batches, start=1):
    print(
        f"Batch {i}: {len(batch)} fixtures"
    )

Liczba batchy: 3
Batch 1: 50 fixtures
Batch 2: 50 fixtures
Batch 3: 32 fixtures


In [ ]:
ingestion_timestamp = datetime.now(timezone.utc)
ingestion_date = ingestion_timestamp.date().isoformat()

output_dir = (
    "/Volumes/wsl_analytics/landing/"
    "sportmonks_raw/"
    "fixture_details/"
    f"league_id={LEAGUE_ID}/"
    f"season_id={SEASON_ID}/"
    f"ingestion_date={ingestion_date}"
)

dbutils.fs.mkdirs(output_dir)

print(output_dir)

/Volumes/wsl_analytics/landing/sportmonks_raw/fixture_details/league_id=45/season_id=26097/ingestion_date=2026-08-13


### 24. Main ingestion loop 

Iterates over batches and for each:
1. Fetches data from the API (`get_fixture_batch`)
2. Writes the raw JSON to `batch_NNN.json` in the Landing directory
3. Records the outcome (`status`, `requested`, `returned`) in `ingestion_results`

The `try/except` block ensures a single-batch failure does not abort the entire run — the failed batch is logged with status `"failed"` and the loop continues.


In [ ]:
ingestion_results = []

for batch_number, batch_ids in enumerate(
    batches,
    start=1
):

    print(
        f"\nBatch {batch_number}/{len(batches)} "
        f"({len(batch_ids)} fixtures)"
    )

    try:

        # 1. Fetches data from the API
        result = get_fixture_batch(
            batch_ids
        )

        returned_fixtures = result.get(
            "data",
            []
        )

        # 2. Landing directory
        output_path = (
            f"{output_dir}/"
            f"batch_{batch_number:03d}.json"
        )

        # 3. Writing raw JSON files
        with open(
            output_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                result,
                f,
                ensure_ascii=False,
                indent=2
            )

        # 4. Log
        ingestion_results.append({
            "batch": batch_number,
            "requested": len(batch_ids),
            "returned": len(returned_fixtures),
            "status": "success",
            "path": output_path,
            "error": None
        })

        print(
            f"✓ Saved "
            f"{len(returned_fixtures)} fixtures"
        )

    except Exception as e:

        ingestion_results.append({
            "batch": batch_number,
            "requested": len(batch_ids),
            "returned": None,
            "status": "failed",
            "path": None,
            "error": str(e)
        })

        print(
            f"✗ ERROR: {e}"
        )

### 25. Ingestion log 
  
Creates a DataFrame from the ingestion results (with an explicit schema for type safety) and displays it. Gives a quick overview of which batches succeeded and which failed.


In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

ingestion_schema = StructType([
    StructField("batch", IntegerType(), False),
    StructField("requested", IntegerType(), False),
    StructField("returned", IntegerType(), True),
    StructField("status", StringType(), False),
    StructField("path", StringType(), True),
    StructField("error", StringType(), True)
])

ingestion_log_df = spark.createDataFrame(
    ingestion_results,
    schema=ingestion_schema
)

display(ingestion_log_df)

batch,requested,returned,status,path,error
1,50,50,success,/Volumes/wsl_analytics/landing/sportmonks_raw/fixture_details/league_id=45/season_id=26097/ingestion_date=2026-08-13/batch_001.json,null
2,50,50,success,/Volumes/wsl_analytics/landing/sportmonks_raw/fixture_details/league_id=45/season_id=26097/ingestion_date=2026-08-13/batch_002.json,null
3,32,32,success,/Volumes/wsl_analytics/landing/sportmonks_raw/fixture_details/league_id=45/season_id=26097/ingestion_date=2026-08-13/batch_003.json,null


### 26. List output files 

Verifies the files were actually written to the Volume.


In [ ]:
display(
    dbutils.fs.ls(output_dir)
)

path,name,size,modificationTime
dbfs:/Volumes/wsl_analytics/landing/sportmonks_raw/fixture_details/league_id=45/season_id=26097/ingestion_date=2026-08-13/batch_001.json,batch_001.json,5162388,1786638600000
dbfs:/Volumes/wsl_analytics/landing/sportmonks_raw/fixture_details/league_id=45/season_id=26097/ingestion_date=2026-08-13/batch_002.json,batch_002.json,5191276,1786638603000
dbfs:/Volumes/wsl_analytics/landing/sportmonks_raw/fixture_details/league_id=45/season_id=26097/ingestion_date=2026-08-13/batch_003.json,batch_003.json,3367923,1786638604000


### 27-28. Coverage check — requested vs returned 

Builds `requested_ids` and `returned_ids` sets, then compares them. A non-empty `Missing` set indicates fixtures the API did not return — require manual investigation or re-fetch.


In [ ]:
requested_ids = set(fixture_ids)

returned_ids = set()

In [ ]:
successful_paths = [
    result["path"]
    for result in ingestion_results
    if result["status"] == "success"
]

In [ ]:
for path in successful_paths:

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        batch_data = json.load(f)

    returned_ids.update(
        fixture["id"]
        for fixture in batch_data.get(
            "data",
            []
        )
    )

In [ ]:
print(
    "Requested:",
    len(requested_ids)
)

print(
    "Returned:",
    len(returned_ids)
)

print(
    "Missing:",
    requested_ids - returned_ids
)

print(
    "Unexpected:",
    returned_ids - requested_ids
)

Requested: 132
Returned: 132
Missing: set()
Unexpected: set()


### 29-30. Pressure coverage check 
  
Checks how many fixtures have pressure data (`n_pressure > 0`) and how many don't. Fixtures without pressure are expected (e.g. limited-coverage matches) but should be identified before downstream analysis.


In [ ]:
fixtures_with_pressure = 0
fixtures_without_pressure = 0

pressure_counts = []

for path in successful_paths:

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        batch_data = json.load(f)

    for fixture in batch_data.get(
        "data",
        []
    ):

        n_pressure = len(
            fixture.get(
                "pressure",
                []
            )
        )

        pressure_counts.append({
            "fixture_id": fixture["id"],
            "fixture_name": fixture["name"],
            "pressure_records": n_pressure
        })

        if n_pressure > 0:
            fixtures_with_pressure += 1
        else:
            fixtures_without_pressure += 1

In [ ]:
print(
    "Fixtures with pressure:",
    fixtures_with_pressure
)

print(
    "Fixtures without pressure:",
    fixtures_without_pressure
)

Fixtures with pressure: 132
Fixtures without pressure: 0


### 31. Pressure coverage DataFrame 

Creates a DataFrame with pressure record counts per fixture, sorted ascending — fixtures with the fewest (including zero) records appear first.


In [ ]:
pressure_check_df = spark.createDataFrame(
    pressure_counts
)

display(
    pressure_check_df
    .orderBy("pressure_records")
)

fixture_id,fixture_name,pressure_records
19499053,Manchester United W vs Chelsea W,178
19499042,Tottenham W vs Manchester City W,182
19499048,West Ham W vs Chelsea W,182
19499094,Manchester City W vs Everton W,182
19499034,Manchester City W vs Brighton W,184
19499044,Brighton W vs West Ham W,184
19499059,London City Lionesses W vs West Ham W,184
19499064,Manchester City W vs West Ham W,184
19499099,Liverpool W vs London City Lionesses W,184
19499056,Tottenham W vs Brighton W,184
